# RAGAS Evaluation: Fireworks AI vs OpenAI

This notebook implements **Activity 1** — comparing our open-source Fireworks AI–powered RAG pipeline against an OpenAI `gpt-4.1-mini`–powered equivalent.

## Objectives

- **Evaluate retrieval quality** — How well does each pipeline retrieve relevant context?
- **Evaluate answer faithfulness** — How well do answers stay grounded in the retrieved context?
- **Evaluate end-to-end accuracy** — How correct are the final answers?
- **Compare cost at scale** — Instrument both pipelines with LangSmith to capture token usage and cost per query.

## Activity 1 Checklist

| Requirement | Status |
|-------------|--------|
| Retrieval quality (context recall) | ✓ RAGAS metrics |
| Answer faithfulness | ✓ Faithfulness metric |
| End-to-end accuracy | ✓ Factual correctness metric |
| Cost & latency comparison | LangSmith traces (see below) |

## Workflow

1. Generate a synthetic evaluation dataset using RAGAS TestsetGenerator.
2. Run both RAG pipelines (Fireworks and OpenAI) on each question.
3. Compute full RAGAS metrics for each provider.
4. Enable LangSmith tracing to capture token usage and latency.
5. Compare results and cost in the LangSmith web console.

## Comparing Cost & Latency in LangSmith

1. **Enable tracing:** Add `LANGCHAIN_TRACING_V2=true` and `LANGCHAIN_API_KEY=<your_key>` to `.env`. Get a key at [smith.langchain.com](https://smith.langchain.com).
2. **Run the notebook** — each `run_rag_pipeline` call is traced automatically.
3. **In the LangSmith web console:**
   - Open your project (e.g. `ragas-fireworks-vs-openai`)
   - **Filter by provider:** Click **Add filter** → **Name** equals `rag-fireworks` or `rag-openai` (or **Metadata** key `provider`, value `fireworks`/`openai`). Use **Traces** tab to see the pipeline runs.
   - **Latency:** Each trace shows total duration; expand to see per-step timing
   - **Token usage:** In each trace, click the LLM span to see input/output token counts
   - **Cost:** Use the **Usage** or **Monitors** tab to see token aggregates; cost = tokens × model pricing (Fireworks vs OpenAI rates)

## Imports and Setup

In [1]:
import os
import copy
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

from app.rag import run_rag_pipeline
from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

# Load .env — try multiple paths (notebook cwd may be workspace root or 16_LLM_Servers)
from pathlib import Path
cwd = Path.cwd()
for p in [cwd / ".env", cwd / "16_LLM_Servers" / ".env"]:
    if p.exists():
        load_dotenv(p, override=True)
        print(f"Loaded .env from {p}")
        break
else:
    load_dotenv(override=True)

# LangSmith: set LANGCHAIN_TRACING_V2=true and LANGCHAIN_API_KEY in .env
os.environ.setdefault("LANGCHAIN_PROJECT", "ragas-fireworks-vs-openai")
tracing = os.environ.get("LANGCHAIN_TRACING_V2", "").lower() == "true"
has_key = bool(os.environ.get("LANGCHAIN_API_KEY"))
if tracing and has_key:
    from langsmith import traceable
    @traceable(name="langsmith-smoke-test")
    def _smoke(): return "ok"
    _smoke()
    print("LangSmith: ON — smoke trace sent. Check project 'ragas-fireworks-vs-openai' at smith.langchain.com")
else:
    print("LangSmith: OFF — add LANGCHAIN_TRACING_V2=true and LANGCHAIN_API_KEY to .env, then re-run this cell")

Loaded .env from /Users/sireeshapulipati/AIE9/16_LLM_Servers/.env
LangSmith: ON — smoke trace sent. Check project 'ragas-fireworks-vs-openai' at smith.langchain.com


## Synthetic Evaluation Dataset 

Use RAGAS TestsetGenerator to create synthetic questions with reference answers and reference contexts. This enables full metrics (context recall, factual correctness, etc.).

In [2]:
# Load and split documents (same corpus as RAG pipeline)
def _tiktoken_len(text: str) -> int:
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(text)
    return len(tokens)

data_dir = os.environ.get("RAG_DATA_DIR", "data")
loader = DirectoryLoader(data_dir, glob="**/*.pdf", loader_cls=PyMuPDFLoader)
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=0, length_function=_tiktoken_len)
docs = splitter.split_documents(raw_docs) if raw_docs else []

# Generator setup (OpenAI for synthetic data creation)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Generate synthetic testset with reference_contexts and reference
# Use generate_with_chunks (not generate_with_langchain_docs) since we have pre-split chunks;
# generate_with_langchain_docs expects full docs and applies HeadlinesExtractor, which fails on PDF chunks
from ragas.testset import TestsetGenerator
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_chunks(chunks=docs, testset_size=5)

print(f"Generated {len(dataset.samples)} evaluation samples (with references)")
dataset.to_pandas().head()

/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_43558/8634776.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_43558/8634776.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


Applying SummaryExtractor:   0%|          | 0/42 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/42 [00:00<?, ?it/s]

Node 21112a72-53f2-458f-9bf4-bc5960940775 does not have a summary. Skipping filtering.
Node 6de48d79-3cb9-454a-8e4f-a881178072f8 does not have a summary. Skipping filtering.
Node b531aece-5de5-4460-b617-d9b2f57889fe does not have a summary. Skipping filtering.
Node 2e1e42d4-b25c-48ec-a8f8-d76114b43fa9 does not have a summary. Skipping filtering.
Node b2eb67d6-bdf1-4ba4-be9d-69f4cadd35d8 does not have a summary. Skipping filtering.
Node 8c10099c-ead2-4ac9-a71e-df25889ecae4 does not have a summary. Skipping filtering.
Node d6bc3aa1-a042-4845-944d-c69d26bccad0 does not have a summary. Skipping filtering.
Node ed8c6d87-cff4-4994-ad1e-fa9638d328fc does not have a summary. Skipping filtering.
Node dba64e56-7b72-4a37-aed3-dec6564b1596 does not have a summary. Skipping filtering.
Node 86fa1028-3e8b-45c1-89bd-9b05c5d537d4 does not have a summary. Skipping filtering.
Node 605b652e-ead5-43bb-b4a4-f288e4ef5b31 does not have a summary. Skipping filtering.
Node 0f396cfc-1fac-42e0-bd3d-ebd3b133c269 d

Applying EmbeddingExtractor:   0%|          | 0/42 [00:00<?, ?it/s]

...

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

Generated 6 evaluation samples (with references)


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,Who is Shannon Gowland and what role she play ...,[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...,"Shannon Gowland, DVM, DABVP, is one of the aut...",Feline Welfare Specialist,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
1,What role did the American Animal Hospital Ass...,[Petcare supported the development of the 2021...,The American Animal Hospital Association (AAHA...,Feline Behavior and Health Consultant,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer
2,how do prescription diets for obesity treatmen...,[<1-hop>\n\ngood starting point is to calculat...,Prescription diets for obesity treatment in ca...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
3,how shud i prepare my kitten for a veterinery ...,[<1-hop>\n\nPractitioners can develop individu...,"To prepare a kitten for a veterinary visit, ve...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
4,How can hypertension in senior cats be identif...,[<1-hop>\n\nConﬂict may occur when a new cat i...,Hypertension in senior cats may manifest as ne...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


## Run Pipelines and Build Evaluation Datasets

For each synthetic question, run both Fireworks and OpenAI pipelines. Update each dataset copy with `response` and `retrieved_contexts` from the pipeline (reference data stays from generator).

In [9]:
# Wrap pipeline calls in explicit traces so LangSmith can filter by provider
from langsmith import traceable

@traceable(name="rag-fireworks", tags=["provider:fireworks"], metadata={"provider": "fireworks"})
def _run_fireworks(question: str):
    return run_rag_pipeline(question, provider="fireworks")

@traceable(name="rag-openai", tags=["provider:openai"], metadata={"provider": "openai"})
def _run_openai(question: str):
    return run_rag_pipeline(question, provider="openai")

# Copy dataset and run Fireworks pipeline on each question
fw_dataset = copy.deepcopy(dataset)
for test_row in fw_dataset:
    result = _run_fireworks(test_row.eval_sample.user_input)
    test_row.eval_sample.response = result["answer"]
    test_row.eval_sample.retrieved_contexts = result["contexts"]

# Copy dataset and run OpenAI pipeline on each question
oai_dataset = copy.deepcopy(dataset)
for test_row in oai_dataset:
    result = _run_openai(test_row.eval_sample.user_input)
    test_row.eval_sample.response = result["answer"]
    test_row.eval_sample.retrieved_contexts = result["contexts"]

# Convert to EvaluationDataset (includes reference, reference_contexts from generator)
# Fill NaN in optional string columns (persona_name, query_style, query_length) to avoid ValidationError
def _prepare_eval_df(df):
    df = df.copy()
    for col in ["persona_name", "query_style", "query_length"]:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)
    return df

fw_eval = EvaluationDataset.from_pandas(_prepare_eval_df(fw_dataset.to_pandas()))
oai_eval = EvaluationDataset.from_pandas(_prepare_eval_df(oai_dataset.to_pandas()))

print(f"Fireworks: {len(fw_eval)} samples | OpenAI: {len(oai_eval)} samples")

Fireworks: 6 samples | OpenAI: 6 samples


## Run RAGAS Evaluation

Evaluate both datasets using full RAGAS metrics (requires reference from synthetic generation). The evaluator uses gpt-4.1-mini as the judge model.

In [10]:
from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0))
metrics = [LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()]
run_config = RunConfig(timeout=360)

# Evaluate Fireworks pipeline
print("Evaluating Fireworks pipeline...")
fw_result = evaluate(dataset=fw_eval, metrics=metrics, llm=evaluator_llm, run_config=run_config)
print("Fireworks:", fw_result)

# Evaluate OpenAI pipeline
print("\nEvaluating OpenAI pipeline...")
oai_result = evaluate(dataset=oai_eval, metrics=metrics, llm=evaluator_llm, run_config=run_config)
print("OpenAI:", oai_result)

/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_43558/3340397017.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_43558/3340397017.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_43558/3340397017.py:2: DeprecationWarning: Importing FactualCorre

Evaluating Fireworks pipeline...


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[35]: TimeoutError()


Fireworks: {'context_recall': 0.8397, 'faithfulness': 0.5868, 'factual_correctness(mode=f1)': 0.4050, 'answer_relevancy': 0.9342, 'context_entity_recall': 0.3757, 'noise_sensitivity(mode=relevant)': 0.1526}

Evaluating OpenAI pipeline...


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[35]: TimeoutError()


OpenAI: {'context_recall': 0.9206, 'faithfulness': 0.8801, 'factual_correctness(mode=f1)': 0.6000, 'answer_relevancy': 0.9585, 'context_entity_recall': 0.4152, 'noise_sensitivity(mode=relevant)': 0.2748}


## Results Summary

| Metric | Fireworks | OpenAI | Winner |
|--------|-----------|--------|--------|
| **Context recall** | 0.8397 | 0.9206 | OpenAI |
| **Faithfulness** | 0.5868 | 0.8801 | OpenAI |
| **Factual correctness (F1)** | 0.4050 | 0.6000 | OpenAI |
| **Answer relevancy** | 0.9342 | 0.9585 | OpenAI |
| **Context entity recall** | 0.3757 | 0.4152 | OpenAI |
| **Noise sensitivity (relevant)** *(lower is better)* | 0.1526 | 0.2748 | Fireworks |

**Summary:** OpenAI `gpt-4.1-mini` leads on most metrics (context recall, faithfulness, factual correctness, answer relevancy, context entity recall). Fireworks scores lower on noise sensitivity (0.16 vs 0.35), which is preferable since lower means less sensitivity to irrelevant context. Overall, the OpenAI pipeline is stronger on retrieval and grounding; Fireworks has room to tune.

**Possible reasons Fireworks underperformed:**
- Embedding model (Qwen3-embedding-8b) retrieves fewer relevant chunks than text-embedding-3-small
- Chat model (gpt-oss-20b) may be less instruction-tuned for strict context grounding
- Different embedding dimensions (4096 vs 1536) and training lead to different retrieval behavior

### LangSmith: Cost and Latency Comparison

| Provider | Latency (per run) | Tokens (per run) | Cost (per run) |
|----------|-------------------|------------------|----------------|
| **Fireworks** | 4–14 s (most ~12 s) | 2,600–5,200 | *Not shown in LangSmith* |
| **OpenAI** | 2–9 s (most 3–5 s) | 3,000–4,200 | ~\$0.0013–\$0.0022 |

**Summary:** OpenAI has lower latency (faster responses, often 2–5 s vs 12 s for Fireworks) and uses fewer tokens per run. Fireworks cost is not surfaced by LangSmith; use [Fireworks pricing](https://fireworks.ai/pricing) with the token counts above to estimate cost. Fireworks is slower mainly because it uses heavier models (Qwen3-embedding-8b for retrieval and gpt-oss-20b for generation) versus OpenAI’s smaller, latency-oriented models.

**Fireworks pipeline traces:**
![Fireworks LangSmith traces](data/fireworks_langsmith.png)

**OpenAI pipeline traces:**
![OpenAI LangSmith traces](data/openai_langsmith.png)

### Fireworks Token Usage

Token consumption from the Fireworks dashboard (serverless usage over time):

![Fireworks token usage](data/fireworks_usage.png)